# IFEST DAC 2026 — V17 (LGBM-23 + C-tower + referensi)
Sel fungsi disalin verbatim dari `proto_v17.py` / `exp10_feat.py` (sumber
tunggal). Official-only, pretrained-free, test-sealed (`docs/rules.md`).
Pertanyaan terbuka: apakah C menambah nilai di atas LGBM-23. Bukan klaim 0.99745 / top-5.


In [2]:
import csv, re, gc, math, json, time, random, hashlib, unicodedata
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from scipy.special import expit

SEED = 42
LT, LV, CC, D = 16, 96, 12, 64
NCH = 37
MARG, EPOCHS, RANK_W = 0.3, 12, 0.2
THR_NN = np.arange(0.2, 0.81, 0.02)
THR_LGB = np.arange(0.05, 0.951, 0.01)
CHARS = {ch: i + 1 for i, ch in enumerate("abcdefghijklmnopqrstuvwxyz0123456789")}
_bce = nn.BCEWithLogitsLoss(reduction="none")
bce = _bce


In [3]:
def reset_rng():
    random.seed(SEED); np.random.seed(SEED)
    import torch
    torch.manual_seed(SEED)

def norm(s):
    s = unicodedata.normalize("NFKC", s or "").lower()
    return re.sub(r"\s+", " ", s).strip()

def toks(s):
    return re.findall(r"[a-z0-9]+", norm(s))

def has_digit(w):
    return bool(re.search(r"\d", w))

def build_groups(rows):
    n = len(rows)
    parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[rb] = ra
    mp_t, mp_c = defaultdict(list), defaultdict(list)
    for i, (_, t, c, _) in enumerate(rows):
        mp_t[norm(t)].append(i); mp_c[norm(c)].append(i)
    for v in list(mp_t.values()) + list(mp_c.values()):
        for j in v[1:]: union(v[0], j)
    return np.array([find(i) for i in range(n)])

def body_toks(t, c):
    tt = toks(t); ct = toks(c)
    lead = ct[:64]
    best, bs = [], -1.0
    st_t = set(tt)
    for s in range(0, max(1, len(ct) - 31), 16):
        w = ct[s:s + 32]
        v = len(st_t & set(w)) / max(len(st_t), 1)
        if v > bs: best, bs = w, v
    return tt, lead + [w for w in best if w not in set(lead)][:32]

def build_vocab(rows, idxs):
    cnt, dfc = Counter(), Counter()
    for i in idxs:
        tt = toks(rows[i][1]) + toks(rows[i][2])
        cnt.update(tt); dfc.update(set(tt))
    VOC = {"<pad>": 0, "<unk>": 1}
    for w, c in cnt.most_common(12000):
        if c >= 2 and w not in VOC: VOC[w] = len(VOC)
    idf = {w: math.log((len(idxs) + 1) / (dfc.get(w, 0) + 1)) + 1.0 for w in VOC}
    return VOC, idf

def encode_rows(rows, BT, VOC, idf):
    HB = 300
    def oov(w): return len(VOC) + (int(hashlib.md5(w.encode()).hexdigest(), 16) % HB)
    n = len(rows)
    Tall = np.zeros((n, LT), int); Vall = np.zeros((n, LV), int)
    DgT = np.zeros((n, LT)); DgV = np.zeros((n, LV)); WtT = np.zeros((n, LT))
    ChT = np.zeros((n, LT, CC), int); ChV = np.zeros((n, LV, CC), int)
    for i in range(n):
        tt, bb = BT[i]
        for L, ws, A, Bg, W, C in ((LT, tt, Tall, DgT, WtT, ChT), (LV, bb, Vall, DgV, None, ChV)):
            ids, dg, wt, ch = [], [], [], []
            for w in ws[:L]:
                ids.append(VOC[w] if w in VOC else oov(w))
                dg.append(1.0 if has_digit(w) else 0.0)
                wt.append(idf.get(w, 1.0))
                ch.append([CHARS.get(x, 0) for x in w[:CC]] + [0] * max(0, CC - len(w)))
            p = L - len(ids)
            A[i] = ids + [0] * p; Bg[i] = dg + [0.0] * p
            if W is not None: W[i] = wt + [0.0] * p
            C[i] = ch + [[0] * CC] * p
    return Tall, Vall, DgT, DgV, WtT, ChT, ChV, len(VOC) + HB

def build_pairs(rows, pool, rng):
    by_body = defaultdict(lambda: [[], []])
    for i in pool:
        by_body[norm(rows[i][2])][rows[i][3]].append(i)
    pairs = []
    for _, (negs, poss) in by_body.items():
        if negs and poss:
            for nn_ in negs:
                pairs.append((rng.choice(poss), nn_))
    return pairs

def cw_for(y, idx):
    p = int((y[idx] == 1).sum())
    return (len(idx) - p) / max(p, 1)

def pick_thr(yv, s, grid):
    from sklearn.metrics import f1_score
    bt, bf = float(grid[0]), -1.0
    for t in grid:
        f = f1_score(yv, (s >= t).astype(int), average="macro", zero_division=0)
        if f > bf: bf, bt = f, float(t)
    return bt, float(bf)

class Net(nn.Module):
    def __init__(self, V):
        super().__init__()
        self.emb = nn.Embedding(V, D, padding_idx=0)
        self.cemb = nn.Embedding(NCH, 16, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(16, 16, k, padding=k // 2) for k in (3, 4, 5)])
        self.cfc = nn.Linear(48, 32)
        self.fc = nn.Sequential(nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 1))

    def tok_rep(self, ids, ch):
        w = self.emb(ids)
        c = self.cemb(ch)
        B, L, _, H = c.shape
        hs = [torch.relu(cv(c.reshape(B * L, CC, H).transpose(1, 2))).amax(dim=2) for cv in self.convs]
        return torch.cat([w, self.cfc(torch.cat(hs, 1)).view(B, L, 32)], -1)

    def forward(self, t, v, ct_, cv_, wt, dt, dv):
        et = self.tok_rep(t, ct_); ev = self.tok_rep(v, cv_)
        etn = et / (et.norm(dim=-1, keepdim=True) + 1e-9)
        evn = ev / (ev.norm(dim=-1, keepdim=True) + 1e-9)
        mt = (t != 0).float(); mv = (v != 0).float()
        cos = (etn @ evn.transpose(1, 2)) * mt.unsqueeze(2) * mv.unsqueeze(1)
        exact = ((t.unsqueeze(2) == v.unsqueeze(1)) & (t.unsqueeze(2) != 0) & (v.unsqueeze(1) != 0)).float()
        dig_eq = ((dt.unsqueeze(2) > 0.5) & (dv.unsqueeze(1) > 0.5)).float() * exact
        padcol = mv.unsqueeze(1)
        feats = []
        for m in (cos, exact, dig_eq):
            mm = m.masked_fill(padcol < 0.5, float("-inf"))
            top2 = mm.topk(min(2, mm.size(2)), dim=2).values
            feats.append(torch.where(torch.isneginf(top2), torch.zeros_like(top2), top2))
        per = torch.cat(feats + [wt.unsqueeze(2), dt.unsqueeze(2)], dim=2)
        real = mt > 0.5
        w = (wt * mt / (wt.sum(1, keepdim=True) + 1e-9)).unsqueeze(2)
        mean = (per * w).sum(1)
        mn = per.masked_fill(~real.unsqueeze(2).expand_as(per), float("inf")).min(dim=1).values
        return self.fc(torch.cat([mean, mn], 1)).squeeze(1)

def _T(x):
    return torch.tensor(x)

def _fit_net(net, E, idxs, y, use_rank, pairs, pw, gpair, epochs=EPOCHS):
    Tall, Vall, DgT, DgV, WtT, ChT, ChV = E
    opt = torch.optim.Adam(net.parameters(), lr=2e-3, weight_decay=1e-5)
    idx = np.arange(len(idxs)); parr = np.array(idxs)
    P = np.array(pairs) if pairs else np.zeros((0, 2), int)
    for ep in range(epochs):
        np.random.shuffle(idx)
        net.train()
        for bb in range(0, len(idx), 256):
            bi = parr[idx[bb:bb + 256]]
            opt.zero_grad()
            out = net(_T(Tall[bi]), _T(Vall[bi]), _T(ChT[bi]), _T(ChV[bi]),
                      _T(WtT[bi]).float(), _T(DgT[bi]).float(), _T(DgV[bi]).float())
            w = torch.where(_T(y[bi]) == 1, torch.tensor(pw), torch.tensor(1.0))
            loss = (_bce(out, _T(y[bi]).float()) * w).mean()
            if len(P):
                pi = torch.randint(0, len(P), (min(128, len(P)),), generator=gpair).numpy()
                pp, qq = P[pi, 0], P[pi, 1]
                sp = net(_T(Tall[pp]), _T(Vall[pp]), _T(ChT[pp]), _T(ChV[pp]),
                         _T(WtT[pp]).float(), _T(DgT[pp]).float(), _T(DgV[pp]).float())
                sn = net(_T(Tall[qq]), _T(Vall[qq]), _T(ChT[qq]), _T(ChV[qq]),
                         _T(WtT[qq]).float(), _T(DgT[qq]).float(), _T(DgV[qq]).float())
                loss = loss + RANK_W * torch.clamp(MARG - (sp - sn), min=0).mean()
            loss.backward(); opt.step()
    return net

def _score_net(net, E, idxs):
    Tall, Vall, DgT, DgV, WtT, ChT, ChV = E
    net.eval()
    with torch.no_grad():
        return torch.sigmoid(net(_T(Tall[idxs]), _T(Vall[idxs]), _T(ChT[idxs]), _T(ChV[idxs]),
                                   _T(WtT[idxs]).float(), _T(DgT[idxs]).float(),
                                   _T(DgV[idxs]).float())).numpy()

def run_nn_fold(rows, y, groups, BT, tro, vao, fi, use_rank):
    """Satu outer-fold utk tower-C. Mengembalikan (skor_outer, threshold_inner)."""
    reset_rng()
    gpair = torch.Generator().manual_seed(SEED + 999)
    prng = random.Random(SEED + 999)
    gtro = groups[tro]
    pos = {idx: k for k, idx in enumerate(tro.tolist())}
    in_s = np.zeros(len(tro))
    for a, b in StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=100 + fi).split(
            np.zeros(len(tro)), y[tro], gtro):
        itr, iva = tro[a], tro[b]
        VOC, idf = build_vocab(rows, itr)
        E = encode_rows(rows, BT, VOC, idf)
        Tall, Vall, DgT, DgV, WtT, ChT, ChV, VSIZE = E
        net = Net(VSIZE).to(torch.device("cpu"))
        E7 = (Tall, Vall, DgT, DgV, WtT, ChT, ChV)
        _fit_net(net, E7, itr, y, use_rank,
                 build_pairs(rows, itr.tolist(), prng) if use_rank else [], cw_for(y, itr), gpair)
        li = [pos[i] for i in iva.tolist()]
        in_s[li] = _score_net(net, E7, iva)
        del net
    thr, _ = pick_thr(y[tro], in_s, THR_NN)
    reset_rng()
    gpair2 = torch.Generator().manual_seed(SEED + 999)
    prng2 = random.Random(SEED + 999)
    VOC, idf = build_vocab(rows, tro)
    E = encode_rows(rows, BT, VOC, idf)
    Tall, Vall, DgT, DgV, WtT, ChT, ChV, VSIZE = E
    net2 = Net(VSIZE).to(torch.device("cpu"))
    E7 = (Tall, Vall, DgT, DgV, WtT, ChT, ChV)
    _fit_net(net2, E7, tro, y, use_rank,
             build_pairs(rows, tro.tolist(), prng2) if use_rank else [], cw_for(y, tro), gpair2)
    return _score_net(net2, E7, vao), float(thr), in_s

def run_lgb_fold(F23, sw8, y, groups, tro, vao, fi):
    """Satu outer-fold utk LGBM-23. Mengembalikan (skor_outer, threshold_inner)."""
    from lightgbm import LGBMClassifier

    def new():
        return LGBMClassifier(objective="binary", n_estimators=450, learning_rate=.03, num_leaves=31,
                              min_child_samples=40, subsample=.90, colsample_bytree=.90,
                              reg_alpha=.2, reg_lambda=2.0, random_state=SEED, n_jobs=-1, verbosity=-1)
    gtro = groups[tro]
    pos = {idx: k for k, idx in enumerate(tro.tolist())}
    in_s = np.zeros(len(tro))
    for a, b in StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=100 + fi).split(
            np.zeros(len(tro)), y[tro], gtro):
        itr, iva = tro[a], tro[b]
        m = new().fit(F23[itr], y[itr], sample_weight=sw8(itr))
        li = [pos[i] for i in iva.tolist()]
        in_s[li] = m.predict_proba(F23[iva])[:, 1]
    thr, _ = pick_thr(y[tro], in_s, THR_LGB)
    m = new().fit(F23[tro], y[tro], sample_weight=sw8(tro))
    return m.predict_proba(F23[vao])[:, 1].astype(np.float32), float(thr), in_s

def run_fold_pair(rows, y, groups, BT, F23, sw8fn, tro, vao, fi, use_rank=True):
    """Satu outer-fold lengkap: inner-OOF kedua tower -> meta+threshold beku -> skor outer.
    Mengembalikan dict skor outer, threshold, dan inner-OOF (utk audit)."""
    from sklearn.linear_model import LogisticRegression
    s_lgb, t_lgb, in_lgb = run_lgb_fold(F23, sw8fn, y, groups, tro, vao, fi)
    s_c, _, in_c = run_nn_fold(rows, y, groups, BT, tro, vao, fi, use_rank)
    meta = LogisticRegression(max_iter=1000, class_weight="balanced")
    meta.fit(np.column_stack([in_lgb, in_c]), y[tro])
    t_bl, _ = pick_thr(y[tro], meta.predict_proba(np.column_stack([in_lgb, in_c]))[:, 1], THR_NN)
    s_bl = meta.predict_proba(np.column_stack([s_lgb, s_c]))[:, 1]
    return {"s_lgb": s_lgb, "s_c": s_c, "s_bl": s_bl, "t_lgb": t_lgb, "t_bl": t_bl,
            "in_lgb": in_lgb, "in_c": in_c}


In [4]:
_re_nw = re.compile(r"[^0-9a-zA-Z]+")
def norm_text(s): return " ".join(_re_nw.sub(" ", str(s).lower()).split())

def toks10(s): return norm_text(s).split()

def ngrams10(x, n): return {" ".join(x[i:i+n]) for i in range(max(0, len(x)-n+1))}

def chargrams10(s, n=3):
    s = norm_text(s).replace(" ", "_")
    return {s[i:i+n] for i in range(max(0, len(s)-n+1))}

def numbers10(s): return re.findall(r"\b\d+(?:[.,]\d+)?\b", str(s).lower())

def proper10(s): return [x.lower() for x in re.findall(r"\b[A-Z][A-Za-z]{2,}\b", str(s))]

def best_window10(tt, ct, win=100, stride=70):
    if not ct: return [], 0.0, 0
    denom = len(set(tt)) or 1.0
    best, bs, bi = [], -1.0, 0
    for st in range(0, max(1, len(ct)), stride):
        ch = ct[st:st+win]
        if not ch: break
        ss = set(ch); sc_ = sum(1 for w in set(tt) if w in ss) / denom
        if sc_ > bs: best, bs, bi = ch, sc_, st
        if st + win >= len(ct): break
    return best, max(bs, 0.0), bi

def row_features10(title, content):
    tt = toks10(title); ct = toks10(content); ts = set(tt); cs = set(ct); lead = ct[:100]
    lset = set(lead); bw, bscore, bpos = best_window10(tt, ct); bset = set(bw)
    denom = max(1, len(ts))
    big = ngrams10(tt, 2); tri = ngrams10(tt, 3); c_norm = " ".join(ct)
    num_t = numbers10(title); num_c = set(numbers10(content))
    neg_t = [w for w in tt if w in NEGS10]; neg_c = set(w for w in ct if w in NEGS10)
    mon_t = [w for w in tt if w in MONTHS10]; mon_c = set(w for w in ct if w in MONTHS10)
    prop = proper10(title); prop_c = set(x.lower() for x in re.findall(r"\b[A-Z][A-Za-z]{2,}\b", str(content)))
    cg = chargrams10(title, 3); cc = chargrams10(content, 3)
    return [len(tt), len(ct), math.log1p(len(ct)),
        sum(w in cs for w in ts) / denom, sum(w in lset for w in ts) / denom, sum(w in bset for w in ts) / denom,
        sum(w in cs for w in ts) / denom, sum(w in lset for w in ts) / denom, bscore, bpos / max(1, len(ct)),
        sum(g in c_norm for g in big) / max(1, len(big)), sum(g in c_norm for g in tri) / max(1, len(tri)),
        len(cg & cc) / max(1, len(cg)), len(num_t), sum(x in num_c for x in num_t) / max(1, len(num_t)),
        sum(x not in num_c for x in num_t), len(neg_t), sum(x not in neg_c for x in neg_t),
        len(mon_t), sum(x not in mon_c for x in mon_t), len(prop),
        sum(x in prop_c for x in prop) / max(1, len(prop)), len(tt) / max(1, len(ct))]

MONTHS10 = ['agu', 'agustus', 'apr', 'april', 'des', 'desember', 'feb', 'februari', 'jan', 'januari', 'jul', 'juli', 'jun', 'juni', 'mar', 'maret', 'mei', 'nov', 'november', 'okt', 'oktober', 'sep', 'september']
NEGS10 = ['batal', 'belum', 'bohong', 'bukan', 'bukanlah', 'gagal', 'hoaks', 'hoax', 'jangan', 'keliru', 'palsu', 'tak', 'tanpa', 'tidak', 'tunda', 'urung']
def sw8(idx):
    c = np.bincount(y[idx], minlength=2); n = len(idx)
    return 8.0 * np.array([n / (2 * max(1, c[0])), n / (2 * max(1, c[1]))])[y[idx]]


In [5]:
def find_csv(name):
    cands = [Path("data") / name, Path.cwd() / "data" / name]
    try:
        kg = Path("/kaggle/input")
        if kg.exists(): cands += sorted(kg.rglob(name))
    except Exception: pass
    for p in cands:
        if p.exists(): return p
    raise FileNotFoundError(name)
TRAIN_P = find_csv("train.csv"); TEST_P = find_csv("test.csv"); SMP_P = find_csv("sample_submission.csv")
try:
    OUT = Path("/kaggle/working/results_v17_nb"); OUT.mkdir(parents=True, exist_ok=True)
except Exception:
    OUT = Path("results/v17_notebook"); OUT.mkdir(parents=True, exist_ok=True)
reset_rng()
rows = []
with open(TRAIN_P, encoding="utf-8") as f:
    for r in csv.DictReader(f):
        rows.append([r["id"], r["title"] or "", r["content"] or "", int(r["label"])])
y = np.array([r[3] for r in rows])
assert len(rows) == 14397, len(rows)
groups = build_groups(rows)
BT = [body_toks(r[1], r[2]) for r in rows]
F23 = np.array([row_features10(r[1], r[2]) for r in rows], dtype=np.float32)
assert F23.shape == (14397, 23), F23.shape
print("data ok: n=%d groups=%d" % (len(rows), len(set(groups.tolist()))), flush=True)


data ok: n=14397 groups=11526



In [6]:
# Nested BERSIH via run_fold_pair (sumber tunggal): inner-OOF -> meta+threshold -> outer.
# Simpan OOF per-baris + threshold per fold.
outer = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y, groups))
OO = {k: np.zeros(len(y)) for k in ("s_lgb", "s_c", "s_bl")}
FLD = np.full(len(y), -1)
rec = {k: [] for k in ("lgb", "blend")}
thrs = {}
t0 = time.time()
for fi, (tro, vao) in enumerate(outer):
    r = run_fold_pair(rows, y, groups, BT, F23, sw8, tro, vao, fi, use_rank=True)
    p_lgb = (r["s_lgb"] >= r["t_lgb"]).astype(int)
    p_bl = (r["s_bl"] >= r["t_bl"]).astype(int)
    OO["s_lgb"][vao] = r["s_lgb"]; OO["s_c"][vao] = r["s_c"]; OO["s_bl"][vao] = r["s_bl"]
    FLD[vao] = fi
    rec["lgb"].append(float(f1_score(y[vao], p_lgb, average="macro", zero_division=0)))
    rec["blend"].append(float(f1_score(y[vao], p_bl, average="macro", zero_division=0)))
    thrs[str(fi)] = {"lgb": r["t_lgb"], "blend": r["t_bl"]}
    del r; gc.collect()
    print("fold %d lgb=%.4f blend=%.4f (%ds)" % (fi, rec["lgb"][-1], rec["blend"][-1], time.time() - t0), flush=True)
for k in ("lgb", "blend"):
    print("CLEAN %s=%.4f %s" % (k, float(np.mean(rec[k])), [round(m, 4) for m in rec[k]]), flush=True)
with open(OUT / "oof_nested.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "y", "fold", "s_lgb", "s_c", "s_bl", "pred_lgb", "pred_bl"])
    for i, (rid, _, _, yy) in enumerate(rows):
        fl = int(FLD[i])
        pl = int(OO["s_lgb"][i] >= thrs[str(fl)]["lgb"]); pb = int(OO["s_bl"][i] >= thrs[str(fl)]["blend"])
        w.writerow([rid, yy, fl, "%.6f" % OO["s_lgb"][i], "%.6f" % OO["s_c"][i], "%.6f" % OO["s_bl"][i], pl, pb])
with open(OUT / "metrics_nested.json", "w", encoding="utf-8") as f:
    json.dump({"lgb": rec["lgb"], "blend": rec["blend"], "thr": thrs}, f, indent=2)
print("nested artifacts saved", flush=True)


fold 0 lgb=0.6093 blend=0.6417 (478s)
fold 1 lgb=0.6093 blend=0.6367 (957s)
fold 2 lgb=0.6088 blend=0.6302 (1439s)
fold 3 lgb=0.6041 blend=0.6122 (1934s)
fold 4 lgb=0.5953 blend=0.6393 (2396s)
CLEAN lgb=0.6054 [0.6093, 0.6093, 0.6088, 0.6041, 0.5953]
CLEAN blend=0.6320 [0.6417, 0.6367, 0.6302, 0.6122, 0.6393]
nested artifacts saved



In [7]:
# Fitting FINAL: cross-fit OOF (vocab/idf per-fold) -> meta + threshold submission.
# Heuristik final; angka performa tetap milik sel nested.
reset_rng()
cf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
Sf_lgb = np.zeros(len(y)); Sf_c = np.zeros(len(y))
for tro, vao in cf.split(np.zeros(len(y)), y, groups):
    mm = LGBMClassifier(objective="binary", n_estimators=450, learning_rate=.03, num_leaves=31,
                        min_child_samples=40, subsample=.90, colsample_bytree=.90,
                        reg_alpha=.2, reg_lambda=2.0, random_state=SEED, n_jobs=-1, verbosity=-1)
    mm.fit(F23[tro], y[tro], sample_weight=sw8(tro))
    Sf_lgb[vao] = mm.predict_proba(F23[vao])[:, 1]
    VOCf, idff = build_vocab(rows, tro)
    Ef = encode_rows(rows, BT, VOCf, idff)
    Tf, Vf, DfT, DfV, WfT, CfT, CfV, VSf = Ef
    prng = random.Random(SEED + 999); gpair = torch.Generator().manual_seed(SEED + 999)
    net = Net(VSf).to(torch.device("cpu"))
    pw = cw_for(y, tro)
    P = np.array(build_pairs(rows, tro.tolist(), prng))
    opt = torch.optim.Adam(net.parameters(), lr=2e-3, weight_decay=1e-5)
    idx = np.arange(len(tro)); parr = np.array(tro)
    for ep in range(EPOCHS):
        np.random.shuffle(idx)
        net.train()
        for bb in range(0, len(idx), 256):
            bi = parr[idx[bb:bb + 256]]
            opt.zero_grad()
            out = net(_T(Tf[bi]), _T(Vf[bi]), _T(CfT[bi]), _T(CfV[bi]),
                      _T(WfT[bi]).float(), _T(DfT[bi]).float(), _T(DfV[bi]).float())
            w = torch.where(_T(y[bi]) == 1, torch.tensor(pw), torch.tensor(1.0))
            loss = (bce(out, _T(y[bi]).float()) * w).mean()
            pi = torch.randint(0, len(P), (min(128, len(P)),), generator=gpair).numpy()
            pp, qq = P[pi, 0], P[pi, 1]
            sp = net(_T(Tf[pp]), _T(Vf[pp]), _T(CfT[pp]), _T(CfV[pp]),
                     _T(WfT[pp]).float(), _T(DfT[pp]).float(), _T(DfV[pp]).float())
            sn = net(_T(Tf[qq]), _T(Vf[qq]), _T(CfT[qq]), _T(CfV[qq]),
                     _T(WfT[qq]).float(), _T(DfT[qq]).float(), _T(DfV[qq]).float())
            loss = loss + RANK_W * torch.clamp(MARG - (sp - sn), min=0).mean()
            loss.backward(); opt.step()
    net.eval()
    with torch.no_grad():
        Sf_c[vao] = torch.sigmoid(net(_T(Tf[vao]), _T(Vf[vao]), _T(CfT[vao]), _T(CfV[vao]),
                                      _T(WfT[vao]).float(), _T(DfT[vao]).float(), _T(DfV[vao]).float())).numpy()
    del mm, net; gc.collect()
metaF = LogisticRegression(max_iter=1000, class_weight="balanced")
metaF.fit(np.column_stack([Sf_lgb, Sf_c]), y)
SFS = metaF.predict_proba(np.column_stack([Sf_lgb, Sf_c]))[:, 1]
TF = max(np.arange(0.2, 0.81, 0.02), key=lambda t: f1_score(y, (SFS >= t).astype(int), average="macro", zero_division=0))
print("final cross-fit blend (optimistik, hanya utk threshold submission): %.4f @t=%.2f" % (
    f1_score(y, (SFS >= TF).astype(int), average="macro", zero_division=0), TF), flush=True)
# Refit kedua tower di FULL train, lalu inferensi test SEKALI.
reset_rng()
VOC, idf = build_vocab(rows, np.arange(len(rows)))
Tall, Vall, DgT, DgV, WtT, ChT, ChV, VS = encode_rows(rows, BT, VOC, idf)
mF = LGBMClassifier(objective="binary", n_estimators=450, learning_rate=.03, num_leaves=31,
                    min_child_samples=40, subsample=.90, colsample_bytree=.90,
                    reg_alpha=.2, reg_lambda=2.0, random_state=SEED, n_jobs=-1, verbosity=-1)
mF.fit(F23, y, sample_weight=sw8(np.arange(len(rows))))
netF = Net(VS).to(torch.device("cpu"))
prng = random.Random(SEED + 999); gpair = torch.Generator().manual_seed(SEED + 999)
Pf = np.array(build_pairs(rows, list(range(len(rows))), prng))
pwF = cw_for(y, np.arange(len(rows)))
optF = torch.optim.Adam(netF.parameters(), lr=2e-3, weight_decay=1e-5)
fidx = np.arange(len(rows))
for ep in range(EPOCHS):
    np.random.shuffle(fidx)
    netF.train()
    for bb in range(0, len(fidx), 256):
        bi = fidx[bb:bb + 256]
        optF.zero_grad()
        out = netF(_T(Tall[bi]), _T(Vall[bi]), _T(ChT[bi]), _T(ChV[bi]),
                   _T(WtT[bi]).float(), _T(DgT[bi]).float(), _T(DgV[bi]).float())
        w = torch.where(_T(y[bi]) == 1, torch.tensor(pwF), torch.tensor(1.0))
        loss = (bce(out, _T(y[bi]).float()) * w).mean()
        pi = torch.randint(0, len(Pf), (min(128, len(Pf)),), generator=gpair).numpy()
        pp, qq = Pf[pi, 0], Pf[pi, 1]
        sp = netF(_T(Tall[pp]), _T(Vall[pp]), _T(ChT[pp]), _T(ChV[pp]),
                  _T(WtT[pp]).float(), _T(DgT[pp]).float(), _T(DgV[pp]).float())
        sn = netF(_T(Tall[qq]), _T(Vall[qq]), _T(ChT[qq]), _T(ChV[qq]),
                  _T(WtT[qq]).float(), _T(DgT[qq]).float(), _T(DgV[qq]).float())
        loss = loss + RANK_W * torch.clamp(MARG - (sp - sn), min=0).mean()
        loss.backward(); optF.step()
print("refit final selesai", flush=True)


final cross-fit blend (optimistik, hanya utk threshold submission): 0.6189 @t=0.32
refit final selesai



In [8]:
test_rows = []
with open(TEST_P, encoding="utf-8") as f:
    for r in csv.DictReader(f):
        test_rows.append([r["id"], r["title"] or "", r["content"] or ""])
assert len(test_rows) == 3603, len(test_rows)
HB = 300
def enc_t(ws, L):
    ids, ch = [], []
    for w in ws[:L]:
        ids.append(VOC[w] if w in VOC else len(VOC) + (int(hashlib.md5(w.encode()).hexdigest(), 16) % HB))
        ch.append([CHARS.get(z, 0) for z in w[:CC]] + [0] * max(0, CC - len(w)))
    p = L - len(ids)
    return ids + [0] * p, ch + [[0] * CC] * p
TEt, TEc_, TEch, TEc_ch, TEd, TEvd, TEw = [], [], [], [], [], [], []
for _, t, x in test_rows:
    tt, bb = body_toks(t, x)
    a, c = enc_t(tt, LT); TEt.append(a); TEch.append(c)
    b, d = enc_t(bb, LV); TEc_.append(b); TEc_ch.append(d)
    TEd.append([1.0 if has_digit(w) else 0.0 for w in tt[:LT]] + [0.0] * max(0, LT - len(tt)))
    TEvd.append([1.0 if has_digit(w) else 0.0 for w in bb[:LV]] + [0.0] * max(0, LV - len(bb)))
    TEw.append([idf.get(w, 1.0) for w in tt[:LT]] + [0.0] * max(0, LT - len(tt)))
TEt = np.array(TEt); TEc_ = np.array(TEc_); TEch = np.array(TEch); TEc_ch = np.array(TEc_ch)
TEd = np.array(TEd); TEvd = np.array(TEvd); TEw = np.array(TEw)
Fte = np.array([row_features10(r[1], r[2]) for r in test_rows], dtype=np.float32)
ste_lgb = mF.predict_proba(Fte)[:, 1]
netF.eval()
with torch.no_grad():
    ste_c = torch.sigmoid(netF(_T(TEt), _T(TEc_), _T(TEch), _T(TEc_ch),
                               _T(TEw).float(), _T(TEd).float(), _T(TEvd).float())).numpy()
ste = metaF.predict_proba(np.column_stack([ste_lgb, ste_c]))[:, 1]
pred = (ste >= TF).astype(int)
pair_map, canonical = {}, {}
tmp_p, tmp_c = defaultdict(set), defaultdict(set)
for _, t, x, yy in rows:
    nt, nc = norm_text(t), norm_text(x)
    tmp_p[(nt, nc)].add(int(yy))
    if int(yy) == 1: tmp_c[nc].add(nt)
for k, v in tmp_p.items():
    if len(v) == 1: pair_map[k] = next(iter(v))
for k, v in tmp_c.items():
    if len(v) == 1: canonical[k] = next(iter(v))
n_ref = 0
for i, (_, t, x) in enumerate(test_rows):
    nt, nc = norm_text(t), norm_text(x)
    if (nt, nc) in pair_map: pred[i] = pair_map[(nt, nc)]; n_ref += 1
    elif nc in canonical: pred[i] = int(nt == canonical[nc]); n_ref += 1
print("reference-overlay: %d/3603" % n_ref, flush=True)
with open(OUT / "submission.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f); w.writerow(["id", "label"])
    for (rid, _, _), p in zip(test_rows, pred):
        w.writerow([rid, int(p)])
with open(OUT / "manifest.json", "w", encoding="utf-8") as f:
    json.dump({"seed": SEED, "thr_final": float(TF), "ref_hits": int(n_ref),
               "towers": "LGBM-23 + C-tower + meta + ref-overlay",
               "external": "none", "pretrained": "none",
               "rules": "official-only, test-sealed, no-pretrained"}, f, indent=2)
print("submission + manifest saved", flush=True)


reference-overlay: 1139/3603
submission + manifest saved



In [9]:
smp = list(csv.DictReader(open(SMP_P, encoding="utf-8")))
sub = list(csv.DictReader(open(OUT / "submission.csv", encoding="utf-8")))
assert len(sub) == 3603 and len(smp) == 3603
assert [r["id"] for r in sub] == [r["id"] for r in smp]
assert set(r["label"] for r in sub) <= {"0", "1"}
print("submission VALID:", dict(Counter(r["label"] for r in sub)), flush=True)
print("VERDICT: promosi hanya bila blend nested > LGBM. Bukan klaim 0.99745/top-5.", flush=True)


submission VALID: {'1': 3111, '0': 492}
VERDICT: promosi hanya bila blend nested > LGBM. Bukan klaim 0.99745/top-5.

